the heat diffusion equation:$$u_t = u_{xx},\quad 0\le x\le 1,\quad t\ge 0$$
with $$u(x,0)=\sin(2\pi x)e^{x},\quad u(0,t)=u(1,t)=0$$
We compute the numerical solution to $T = 1$, and determine the order of convergence.




Let $x_j = jh$ with $h = 1/(N+1)$,internal nodes $j=1…N$\
The Laplacian matrix eigenvalues (DST-I):$$\lambda_k = -\frac{4}{h^2}\sin^2\left(\frac{k\pi}{2(N+1)}\right)$$
This avoids forming any matrices.\
**(a)Finite Difference + Fast Sine Transform**\
we usee backward-Euler:$$\frac{u^{n+1} - u^n}{\Delta t} = u^{n+1}_{xx}
\Rightarrow
(I - \Delta t A)u^{n+1} = u^n$$
FST diagonalizes A → solve in Fourier space:$$\hat u_k^{n+1} = \frac{\hat u_k^n}{1 - \Delta t \lambda_k}$$


In [1]:
pip install numpy scipy


In [2]:
import numpy as np
from scipy.fft import dst, idst

def heat_FDM_FST(N, T, dt):
    """
    Solves u_t = u_xx using implicit finite difference + FST
    Dirichlet BC . N=number of internal nodes
    """
    h = 1/(N+1)
    x = np.linspace(h, 1-h, N)

    # Initial condition
    u = np.sin(2*np.pi*x)*np.exp(x)

    # eigenvalues of Laplacian
    k = np.arange(1, N+1)
    lam = -4/h**2*np.sin(np.pi*k/(2*(N+1)))**2

    steps = int(T/dt)

    # spectral variable
    u_hat = dst(u, type=1)

    for _ in range(steps):
        u_hat = u_hat/(1 - dt*lam)

    u = idst(u_hat, type=1)/(2*(N+1))
    return x, u


**(b) Method of Lines (MOL)**\
We discretize only space:$$u_{xx}(x_j) \approx \frac{u_{j-1} - 2u_j + u_{j+1}}{h^2}$$
This gives an ODE system:$$\dot{u} = Au$$
We solve it by explicit RK4

In [7]:
def heat_MOL(N, T, dt):
    h = 1/(N+1)
    x = np.linspace(h,1-h,N)

    def lap(u):
        out = np.zeros_like(u)
        out[1:-1] = (u[:-2] - 2*u[1:-1] + u[2:])/h**2
        out[0] = ( -2*u[0] + u[1])/h**2
        out[-1] = (u[-2] -2*u[-1])/h**2
        return out

    u = np.sin(2*np.pi*x)*np.exp(x)
    steps = int(T/dt)

    for _ in range(steps):
        k1 = lap(u)
        k2 = lap(u + dt*k1/2)
        k3 = lap(u + dt*k2/2)
        k4 = lap(u + dt*k3)
        u = u + dt*(k1 + 2*k2 + 2*k3 + k4)/6
    return x,u


**Order of Convergence**
Strategy:\
1.Compute solutions for mesh sizes
$N,2N,4N$

2.Compare norm of differences
Use ratio method:$$p = \log_2\frac{\|u_{N}-u_{2N}\|}{\|u_{2N}-u_{4N}\|}$$

In [8]:
def error_norm(u1,u2):
    return np.linalg.norm(u1-u2,2)

def convergence_test(solver):
    T=1
    dt=1e-3
    Ns=[50,100,200]
    sol=[]
    for N in Ns:
        _,u=solver(N,T,dt)
        sol.append(u)

    e1=error_norm(sol[0],sol[1][::2])
    e2=error_norm(sol[1],sol[2][::2])
    p=np.log2(e1/e2)
    return e1,e2,p


Hence,If p ≈ 1 → time (BE) dominated.\
and If p ≈ 2 → spatial dominated (or RK4+MOL case).